In [129]:
import numpy as np
import time
import copy

In [130]:
state_num = 3**9
action_num = 9
gamma = 0.99

In [131]:
def ternary_to_decimal(ternary_str: str) -> int:
    """将三进制字符串转换为十进制整数"""
    if not ternary_str or not all(char in '012' for char in ternary_str):
        raise ValueError("输入必须是仅包含 0, 1, 2 的非空字符串")
    return int(ternary_str, 3)

def decimal_to_ternary(decimal_num: int, length: int = 9) -> str:
    """
    将十进制整数转换为指定位数的三进制字符串，不足补0
    :param decimal_num: 十进制整数
    :param length: 目标字符串长度，默认为 9
    :return: 补0后的三进制字符串
    """
    # 1. 处理特殊值 0
    if decimal_num == 0:
        return "0".zfill(length)
    
    # 2. 处理负数
    is_negative = False
    if decimal_num < 0:
        is_negative = True
        decimal_num = abs(decimal_num)
    
    # 3. 核心逻辑：除 3 取余，逆序排列
    ternary_digits = []
    while decimal_num > 0:
        remainder = decimal_num % 3
        ternary_digits.append(str(remainder))
        decimal_num //= 3
    
    # 4. 逆序拼接字符串
    result = ''.join(reversed(ternary_digits))
    
    # 5. 【核心需求】使用 zfill 在前面补 0 到指定长度
    result = result.zfill(length)
    
    # 如果是负数，补上负号（注意：负号会额外占用一个字符位置）
    return '-' + result if is_negative else result

def check_winner(board: str) -> int:
    """
    检查井字棋的胜负状态。
    
    参数:
        board (str): 长度为9的字符串，'0'为空，'1'为X，'2'为O。
        
    返回:
        int: 0表示平局或无人赢，1表示X赢，2表示O赢。
    """
    if len(board) != 9:
        raise ValueError("棋盘字符串长度必须为9")
    
    # 井字棋所有的获胜组合（行、列、对角线对应的索引）
    win_patterns = [
        (0, 1, 2), (3, 4, 5), (6, 7, 8),  # 3行
        (0, 3, 6), (1, 4, 7), (2, 5, 8),  # 3列
        (0, 4, 8), (2, 4, 6)              # 2条对角线
    ]
    
    for a, b, c in win_patterns:
        # 如果这三个位置都有棋子，且棋子相同，且不是空位('0')
        if board[a] == board[b] == board[c] and board[a] != '0':
            return int(board[a])  # 返回 '1' 或 '2'
            
    return 0  # 没有人赢

In [132]:
def reward_model_TicTacToe(state_index, action, side, current_action_side):
    reward = None
    state_vec = decimal_to_ternary(state_index)
    
    # 先检查有没有进入吸收态
    if state_index == state_num:
        reward = 0
    # 再检查目前step是否可行动
    elif current_action_side != side:
        reward = 0
    # 再检查当棋盘上仍有位置时，当前动作是否下在了已有棋子的位置
    elif state_vec.count('0') != 0 and state_vec[action] != '0':
        reward = -3
    # 再检查对方是否已经获胜
    elif str(check_winner(state_vec)) != '0' and str(check_winner(state_vec)) != side:
        reward = -1
    # 动作有效
    else:
        state_vec_list = list(state_vec)
        state_vec_list[action] = side
        state_vec = "".join(state_vec_list)
        # 获胜
        if str(check_winner(state_vec)) == side:
            reward = 1
        # 进行中
        else:
            reward = 0

    return reward

def env_model_TicTacToe(state_index, action, side):
    next_state_index = None
    state_vec = decimal_to_ternary(state_index)
    
     # 先检查有没有进入吸收态
    if state_index == state_num:
        next_state_index = state_num
    # 再检查是否已经有一方获胜
    elif str(check_winner(state_vec)) != '0' or state_vec.count('0') == 0:
        next_state_index = state_num
    # 再检查是否下在了已有棋子的位置
    elif state_vec[action] != '0':
        next_state_index = state_index
    # 动作有效
    else:
        state_vec_list = list(state_vec)
        state_vec_list[action] = side
        state_vec = "".join(state_vec_list)
        next_state_index = ternary_to_decimal(state_vec)

    return next_state_index

In [133]:
# 纯loop求和迭代版本    
# 初始化
value_table_dict = dict()
value_table_dict['1'] = np.zeros(state_num + 1)
value_table_dict['2'] = np.zeros(state_num + 1)
policy_table = np.full((state_num + 1, 9), 1/9)

# 策略评价迭代上限
j_truncate = 10
# 策略改进迭代上限
k_truncate = 10

start_time = time.time()
for k in range(k_truncate):
    old_policy_table = copy.deepcopy(policy_table)
    current_side = str(k % 2 + 1)
    # 策略评价迭代
    for j in range(j_truncate):
        # 对每个状态进行更新
        for s in range(state_num + 1):
            v_s = 0
            state_vec = decimal_to_ternary(s)
            current_action_side = '1' if state_vec.count('1') == state_vec.count('2') else '2'
            for a in range(action_num):
                r_step = reward_model_TicTacToe(s, a, current_side, current_action_side)
                s_next = env_model_TicTacToe(s, a, current_action_side)
                v_s_next = value_table_dict[current_side][s_next]
                action_prob = policy_table[s, a]            
                v_s = v_s + action_prob * (r_step + gamma * v_s_next)
                # if s == 16440:
                #     print(f"v_s:{v_s}, a:{a}, r:{r_step}, a_p:{action_prob}")

            value_table_dict[current_side][s] = v_s
    # 策略改进
    for s in range(state_num + 1):
        q_s = np.zeros(action_num)
        state_vec = decimal_to_ternary(s)
        current_action_side = '1' if state_vec.count('1') == state_vec.count('2') else '2'
        for a in range(action_num):
            r_step = reward_model_TicTacToe(s, a, current_action_side, current_action_side)
            s_next = env_model_TicTacToe(s, a, current_action_side)
            v_s_next = value_table_dict[current_action_side][s_next]
            q_s[a] = r_step + gamma * v_s_next
            
        opt_a_index = np.argmax(q_s)
        policy_table[s, :] = 0
        policy_table[s, opt_a_index] = 1
    
    is_policy_changed = not np.array_equal(old_policy_table, policy_table) 
    print(f"The {k+1}-th iteration finished, policy change{is_policy_changed}")


end_time = time.time()    # 记录结束时间
elapsed_time = end_time - start_time

print(f"循环执行耗时: {elapsed_time:.3f} 秒")

The 1-th iteration finished, policy changeTrue
The 2-th iteration finished, policy changeTrue
The 3-th iteration finished, policy changeTrue
The 4-th iteration finished, policy changeTrue
The 5-th iteration finished, policy changeTrue
The 6-th iteration finished, policy changeTrue
The 7-th iteration finished, policy changeFalse
The 8-th iteration finished, policy changeFalse
The 9-th iteration finished, policy changeFalse
The 10-th iteration finished, policy changeFalse
循环执行耗时: 68.751 秒


In [148]:
np.save('./model/model_TicTacToe_v1.npy', policy_table)

In [149]:
# 后续加载回来
loaded_arr = np.load('./model/model_TicTacToe_v1.npy')

In [150]:
loaded_arr

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.]])